# Spatial Filtering Experiments
Gaussian LPF and Unsharp Masking parameter sweeps with histogram comparisons.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from preprocessing.dataset_loader import DatasetLoader
from filtering_spatial.gaussian_lpf import GaussianLPF
from filtering_spatial.unsharp_masking import UnsharpMasking
from filtering_spatial.spatial_filter import SpatialFilter

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '02_spatial_filtering'
PREPROC_DIR = PROJECT_ROOT / 'output' / '01_preprocessing' / 'normalized'

In [ ]:
import cv2

# Load a sample preprocessed image
sample_paths = sorted(PREPROC_DIR.glob('*.png'))
assert len(sample_paths) > 0, 'Run preprocessing notebook first!'
sample = cv2.imread(str(sample_paths[0]), cv2.IMREAD_GRAYSCALE)
print(f'Sample shape: {sample.shape}')

In [ ]:
# Side-by-side: original vs. gaussian vs. unsharp
sf = SpatialFilter()
comparison = sf.compare_both(sample)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (name, img) in zip(axes, comparison.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(name, fontsize=11)
    ax.axis('off')
plt.suptitle('Spatial Filter Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comparison_original_gaussian_unsharp.png', dpi=150)
plt.show()

In [ ]:
# Gaussian parameter sweep (kernel size)
glpf = GaussianLPF()
kernel_sizes = [3, 5, 7, 11, 15]
sigmas = [1.0, 2.0]

fig, axes = plt.subplots(len(sigmas), len(kernel_sizes), figsize=(16, 6))
for row, sigma in enumerate(sigmas):
    for col, k in enumerate(kernel_sizes):
        out = glpf.apply_with_params(sample, k, sigma)
        axes[row, col].imshow(out, cmap='gray')
        axes[row, col].set_title(f'k={k}, σ={sigma}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Gaussian LPF Parameter Sweep', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gaussian_param_sweep.png', dpi=150)
plt.show()

In [ ]:
# Unsharp masking parameter sweep
um = UnsharpMasking()
radii = [0.5, 1.0, 2.0]
amounts = [0.5, 1.0, 1.5, 2.0]

fig, axes = plt.subplots(len(radii), len(amounts), figsize=(16, 10))
for row, r in enumerate(radii):
    for col, a in enumerate(amounts):
        out = um.apply_with_params(sample, r, a, 0)
        axes[row, col].imshow(out, cmap='gray')
        axes[row, col].set_title(f'r={r}, a={a}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Unsharp Masking Parameter Sweep', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'unsharp_param_sweep.png', dpi=150)
plt.show()

In [ ]:
# Histogram comparison
gaussian_out = sf.apply_gaussian(sample, 5, 1.0)
unsharp_out = sf.apply_unsharp(sample, 1.0, 1.0, 0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, img) in zip(axes, [
    ('Original', sample),
    ('Gaussian LPF (k=5, σ=1)', gaussian_out),
    ('Unsharp Mask (r=1, a=1)', unsharp_out)
]):
    ax.hist(img.ravel(), bins=64, color='steelblue', alpha=0.7)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Pixel value')
plt.suptitle('Histogram Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histogram_comparison.png', dpi=150)
plt.show()

In [ ]:
# Batch process all preprocessed images
all_paths = sorted(PREPROC_DIR.glob('*.png'))
print(f'Processing {len(all_paths)} images...')

saved_gaussian = sf.batch_gaussian(all_paths, OUTPUT_DIR, kernel_size=5, sigma=1.0)
saved_unsharp = sf.batch_unsharp(all_paths, OUTPUT_DIR, radius=1.0, amount=1.0)
print(f'Gaussian: {len(saved_gaussian)} saved')
print(f'Unsharp: {len(saved_unsharp)} saved')